# mini-sglang on a GPU: attention backends, CUDA Graphs, benchmarks

Runs on Colab (T4 or A100) or any CUDA machine. It

1. installs this repo (branch `claude/gpu-path`) and downloads `Qwen/Qwen3-0.6B`,
2. runs the **whole test suite on the GPU** (incl. `tests/test_gpu.py`, which is skipped on CPU: real Triton kernels, CUDA Graph capture/replay, fp32 greedy == HuggingFace),
3. runs **every script in `benchmarks/`** with the `torch` and `triton` attention backends, with and without CUDA Graphs,
4. records a **`torch.profiler` trace of one decode step** for each configuration.

Nothing here has been run yet: the output cells are intentionally empty. Each section has a *What to expect* note so you can tell whether the numbers make sense.

> **T4 note**: the T4 (sm_75) has no bfloat16 tensor cores, so the notebook uses `float16` there and `bfloat16` on Ampere+ (A100, L4, ...).

## 0. Setup

In [ ]:
!nvidia-smi

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/willychang21/mini-sglang"
BRANCH = "claude/gpu-path"
if not os.path.exists("mini-sglang"):
    !git clone -b {BRANCH} {REPO_URL}
%cd mini-sglang
!git log --oneline -3
# Colab already ships a CUDA build of torch (with triton). Do not reinstall torch.
!pip install -q -e ".[test]" pyzmq
!python -c "import torch, triton; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'triton', triton.__version__)"

In [ ]:
from huggingface_hub import snapshot_download
MODEL = "/content/models/Qwen3-0.6B" if os.path.exists("/content") else os.path.abspath("models/Qwen3-0.6B")
snapshot_download("Qwen/Qwen3-0.6B", local_dir=MODEL)
os.environ["MINISGL_TEST_MODEL"] = MODEL  # enables tests/test_real_model.py and the real-model GPU test

import torch
major, minor = torch.cuda.get_device_capability()
DTYPE = "bfloat16" if major >= 8 else "float16"
print(torch.cuda.get_device_name(), f"sm_{major}{minor}", "->", DTYPE)

## 1. Test suite on the GPU

`tests/test_gpu.py` only runs here. It checks

* CUDA Graph decode (triton backend, fp32, page_size 1 and 16) reproduces HuggingFace greedy output while the batch shrinks from 6 to 1 requests (padded to 8/4/2/1 with the dummy request),
* eager torch / triton backends on CUDA == HuggingFace,
* bf16 graph replay == bf16 eager,
* real Qwen3-0.6B, fp32, triton + CUDA Graph == HuggingFace.

`tests/test_triton_attention.py` now compiles the kernels for the GPU instead of running them in the Triton interpreter.

*What to expect*: everything passes; `test_tp.py` still uses Gloo on CPU.

In [ ]:
!python -m pytest -q -x -rs

## 2. Benchmarks

Every script takes `--device cuda --dtype ... --attention-backend {torch,triton} [--disable-cuda-graph]`.
The `torch` backend has no CUDA Graph support (its metadata shapes change every step), so it always runs eagerly.

In [ ]:
CONFIGS = [  # (label, extra flags)
    ("torch, eager",          "--attention-backend torch --disable-cuda-graph"),
    ("triton, eager",         "--attention-backend triton --disable-cuda-graph"),
    ("triton, CUDA graph",    "--attention-backend triton"),
]
COMMON = f"--model {MODEL} --device cuda --dtype {DTYPE}"

def run_all(script, extra=""):
    for label, flags in CONFIGS:
        print(f"\n######## {script} | {label}\n", flush=True)
        !python benchmarks/{script} {COMMON} {flags} {extra}

### 2.1 Attention kernels + end-to-end decode (`bench_attention.py`)

Kernel tables: one attention call for one layer; torch = gather through the page table into a padded tensor + SDPA; triton = paged kernels, decode with `num_kv_splits` = 1/4/8/16.
End-to-end table: decode TPOT through the scheduler for torch / triton eager / triton + CUDA Graph.

*What to expect*
* decode: triton beats gather+SDPA, increasingly with kv_len (the gather copies all of K/V every layer); for bs=1 and long kv_len, splits > 1 is clearly faster than splits = 1 (one program per KV head cannot fill the GPU); for bs=64 splits matter little.
* prefill: torch SDPA (FlashAttention / mem-efficient kernels) may well win against this simple extend kernel; that is fine, prefill is compute-bound.
* end to end: CUDA Graph cuts decode TPOT a lot at small batch (Qwen3-0.6B decode is launch-bound: ~28 layers x ~15 kernels); the gap shrinks as bs grows.

In [ ]:
!python benchmarks/bench_attention.py {COMMON}

### 2.2 Ch4 KV cache (`bench_kv_cache.py`)
Uses `SequenceRunner` (prefill-phase batches), so CUDA Graph does not apply; the backend does.

*What to expect*: KV cache speedup grows with output length; `same output` is True.

In [ ]:
run_all("bench_kv_cache.py")

### 2.3 Ch6/7 continuous batching (`bench_batching.py`)
*What to expect*: continuous batching gives several x the throughput of one-at-a-time; CUDA Graph helps the one-at-a-time row most (bs = 1 decode is purely launch-bound). `identical outputs` should be True in fp32; in bf16/fp16 an occasional difference between batch sizes is possible (different GEMM shapes, different rounding).

In [ ]:
run_all("bench_batching.py")

### 2.4 Ch8 prefix cache (`bench_prefix_cache.py`)
*What to expect*: radix hit rate > 90%, much lower TTFT than naive; identical outputs.

In [ ]:
run_all("bench_prefix_cache.py")

### 2.5 Ch10 speculative decoding (`bench_speculative.py`)
Uses `SequenceRunner`, so no CUDA Graph; compare torch vs triton only.

*What to expect*: "self (upper bound)" accepts 100%; layer-truncated drafts accept less; `lossless` True in fp32 (bf16 verification of k tokens at once vs one at a time can differ in rare near-ties).

In [ ]:
run_all("bench_speculative.py", "--draft-layers 7,14")

### 2.6 Ch5 serving (`bench_serving.py`)
Starts the HTTP server once per configuration and drives it at several concurrencies.

*What to expect*: TPOT roughly flat with concurrency for triton + CUDA Graph until the batch gets large; higher with eager backends.

In [ ]:
import time, subprocess, httpx

SERVER_FLAGS = {
    "torch, eager":       ["--attention-backend", "torch", "--disable-cuda-graph"],
    "triton, eager":      ["--attention-backend", "triton", "--disable-cuda-graph"],
    "triton, CUDA graph": ["--attention-backend", "triton"],
}
for label, flags in SERVER_FLAGS.items():
    print(f"\n######## bench_serving.py | {label}\n", flush=True)
    server = subprocess.Popen([sys.executable, "-m", "minisgl", "--model", MODEL, "--device", "cuda",
                               "--dtype", DTYPE, "--port", "1919", *flags],
                              stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    try:
        for _ in range(300):  # wait until ready (graph capture takes a few seconds)
            try:
                if httpx.get("http://127.0.0.1:1919/v1/models", timeout=1).status_code == 200:
                    break
            except Exception:
                time.sleep(1)
        !python benchmarks/bench_serving.py --url http://127.0.0.1:1919 --concurrency 1,4,16 --num-requests 32
    finally:
        server.terminate(); server.wait()

## 3. `torch.profiler` trace of one decode step

We wrap `Engine.forward_batch` and profile exactly one decode step (batch of 8 requests, ~256 tokens of context) for each configuration, print the top CUDA kernels and save a Chrome trace (open in `chrome://tracing` or https://ui.perfetto.dev).

*What to expect*
* torch backend: per layer an `index` (gather) of the whole K/V history, mask construction, then an SDPA kernel;
* triton eager: `_decode_stage1_kernel` + `_decode_merge_kernel` per layer and no gather; many small launches with CPU gaps between them;
* triton + CUDA graph: the CPU side is a single `cudaGraphLaunch`, the GPU timeline is dense (no gaps).

In [ ]:
import torch
from torch.profiler import profile, ProfilerActivity
from minisgl import SamplingParams
from minisgl.llm import LLM

def profile_one_decode_step(backend, disable_graph, trace_path):
    llm = LLM(MODEL, load_tokenizer_=False, device="cuda", dtype=DTYPE, attention_backend=backend,
              disable_cuda_graph=disable_graph, max_total_tokens=16384, max_seq_len=2048, max_running_req=16)
    engine = llm.engine
    llm.generate([[1, 2, 3]] * 2, SamplingParams(max_tokens=4))  # warm-up / JIT
    orig = engine.forward_batch
    state = {"seen": 0, "prof": None}

    def wrapped(batch, args):
        if batch.is_decode and state["prof"] is None:
            state["seen"] += 1
            if state["seen"] == 3:  # skip the first decode steps
                torch.cuda.synchronize()
                with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
                    out = orig(batch, args)
                    torch.cuda.synchronize()
                state["prof"] = prof
                return out
        return orig(batch, args)

    engine.forward_batch = wrapped
    g = torch.Generator().manual_seed(0)
    prompts = torch.randint(100, 10000, (8, 256), generator=g).tolist()
    llm.generate(prompts, SamplingParams(max_tokens=8, ignore_eos=True))
    prof = state["prof"]
    prof.export_chrome_trace(trace_path)
    print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=15))
    llm.shutdown()
    return prof

for backend, disable_graph in [("torch", True), ("triton", True), ("triton", False)]:
    name = f"decode_{backend}_{'eager' if disable_graph else 'graph'}"
    print(f"\n######## {name}\n")
    profile_one_decode_step(backend, disable_graph, f"/content/{name}.json" if os.path.exists("/content") else f"{name}.json")

In [ ]:
# Download the traces (Colab)
try:
    from google.colab import files
    for n in ["decode_torch_eager", "decode_triton_eager", "decode_triton_graph"]:
        files.download(f"/content/{n}.json")
except ImportError:
    print("not on Colab: traces are in the working directory")

## 4. Notes to write back into the README

* Paste the tables from 2.1 (kernels + e2e) and the TPOT rows of 2.3 into `README.md` under a "GPU" section.
* If any `test_gpu.py` test fails, the most informative ones are `test_eager_fp32_matches_hf[triton]` (kernels) vs `test_cuda_graph_fp32_matches_hf` (graph padding / static buffers).